In [14]:
import sqlite3
import pandas as pd
from BQ_dataset import create_client
from config import PROJECT_ID,DATASET_ID

In [15]:
client=create_client()

In [16]:
# Lista de tablas
tablas = [
    "channels", "categories", "costs", "product_status", "timeline", 
    "order_status", "payment_status", "customers", "products", 
    "orders", "order_items", "payments", "reviews"
]

# 2. Crear la conexión a SQLite en memoria
conn = sqlite3.connect(':memory:')

# 3. Descargar cada tabla de BQ e insertarla en SQLite
print("Descargando tablas desde BigQuery a SQLite...")
for tabla in tablas:
    query_bq = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{tabla}`"
    # Descargamos a Pandas
    df_temporal = client.query(query_bq).to_dataframe()
    
    # Lo enviamos a SQLite (si ya existe, la reemplaza)
    df_temporal.to_sql(tabla, conn, index=False, if_exists='replace')
    print(f" -> Tabla '{tabla}' lista en SQLite ({len(df_temporal)} filas).")


Descargando tablas desde BigQuery a SQLite...


c:\Users\guill\Documents\UC3M\ia\Repos_Visual\Jack-attack\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 -> Tabla 'channels' lista en SQLite (4 filas).


c:\Users\guill\Documents\UC3M\ia\Repos_Visual\Jack-attack\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 -> Tabla 'categories' lista en SQLite (70 filas).


...

c:\Users\guill\Documents\UC3M\ia\Repos_Visual\Jack-attack\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 -> Tabla 'order_items' lista en SQLite (5063 filas).


c:\Users\guill\Documents\UC3M\ia\Repos_Visual\Jack-attack\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 -> Tabla 'payments' lista en SQLite (2000 filas).
 -> Tabla 'reviews' lista en SQLite (1676 filas).


c:\Users\guill\Documents\UC3M\ia\Repos_Visual\Jack-attack\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [17]:
query_1 = """
SELECT 
    strftime('%Y-%m', t.pedido) AS mes,
    COUNT(DISTINCT o.id) AS total_pedidos,
    ROUND(SUM(oi.cantidad * oi.precio_compra * (1 - COALESCE(oi.descuento, 0))), 2) AS ingresos_totales,
    ROUND(SUM((oi.cantidad * oi.precio_compra * (1 - COALESCE(oi.descuento, 0))) - (oi.cantidad * c.coste)), 2) AS beneficio_neto
FROM orders o
JOIN timeline t ON o.timeline_id = t.id
JOIN order_items oi ON o.id = oi.pedido_id
JOIN products p ON oi.producto_id = p.id
JOIN costs c ON p.coste_id = c.id
JOIN order_status os ON o.estado_pedido_id = os.id
WHERE os.estado != 'Cancelado'
GROUP BY mes
ORDER BY mes DESC;
"""
df_ingresos = pd.read_sql_query(query_1, conn)
df_ingresos


,mes,total_pedidos,ingresos_totales,beneficio_neto
0,2025-09,8,33400.50,4794.36
1,2025-08,50,272257.31,118459.69
2,2025-07,64,347958.30,107100.90
3,2025-06,37,180611.20,74606.08
4,2025-05,63,284329.73,130531.76
5,2025-04,55,306943.92,116895.17
6,2025-03,47,228189.69,120919.98
7,2025-02,50,241126.86,79931.96
8,2025-01,62,305011.11,92350.06
9,2024-12,47,231870.46,85249.89


In [18]:
query_2 = """
SELECT 
    p.id AS producto_id,
    cat.nombre AS categoria,
    SUM(oi.cantidad) AS unidades_vendidas,
    ROUND(AVG(r.rating), 2) AS valoracion_media,
    COUNT(r.id) AS total_reseñas
FROM order_items oi
JOIN products p ON oi.producto_id = p.id
JOIN categories cat ON p.categoria_id = cat.id
LEFT JOIN reviews r ON oi.id = r.linea_pedido_id
GROUP BY p.id
ORDER BY unidades_vendidas DESC
LIMIT 10;
"""
df_top_productos = pd.read_sql_query(query_2, conn)
df_top_productos


,producto_id,categoria,unidades_vendidas,valoracion_media,total_reseñas
0,8,Quantum Studio,284,3.16,25
1,6,Krypton Prime,272,2.75,20
2,62,Helix Max,271,3.24,29
3,60,Cortex Air,270,2.56,32
4,1,Astra Neo,270,2.86,29
5,46,Vector Pro,258,3.28,25
6,68,Aura Prime,257,3.04,26
7,64,Volt Carbon,257,3.29,17
8,66,Zeta Prime,254,2.89,37
9,49,Quantum Prime,254,3.30,20


In [19]:
query_3 = """
SELECT 
    os.estado AS estado_actual_pedido,
    COUNT(o.id) AS cantidad_pedidos,
    ROUND(AVG(julianday(t.envio) - julianday(t.pedido)), 1) AS dias_medios_envio,
    ROUND(AVG(julianday(t.entrega) - julianday(t.envio)), 1) AS dias_medios_transito,
    ROUND(AVG(julianday(t.entrega) - julianday(t.pedido)), 1) AS dias_totales_entrega
FROM orders o
JOIN timeline t ON o.timeline_id = t.id
JOIN order_status os ON o.estado_pedido_id = os.id
WHERE t.entrega IS NOT NULL
GROUP BY os.estado;
"""
df_logistica = pd.read_sql_query(query_3, conn)
df_logistica


,estado_actual_pedido,cantidad_pedidos,dias_medios_envio,dias_medios_transito,dias_totales_entrega
0,delivered,988,230.0,233.2,463.3
1,returned,1012,237.2,234.3,471.5


In [20]:
query_4 = """
SELECT 
    c.pais,
    COUNT(DISTINCT c.id) AS total_clientes,
    COUNT(DISTINCT o.id) AS total_pedidos,
    ROUND(AVG(oi.cantidad * oi.precio_compra), 2) AS ticket_medio_linea
FROM customers c
LEFT JOIN orders o ON c.id = o.cliente_id
LEFT JOIN order_items oi ON o.id = oi.pedido_id
GROUP BY c.pais
ORDER BY total_clientes DESC;
"""
df_paises = pd.read_sql_query(query_4, conn)
df_paises


,pais,total_clientes,total_pedidos,ticket_medio_linea
0,Francia,136,506,3975.90
1,Alemania,127,509,3795.64
2,Italia,120,512,3888.90
3,España,117,473,3906.89


In [21]:
query_5 = """
SELECT 
    ch.canal AS canal_adquisicion,
    COUNT(DISTINCT cust.id) AS clientes_captados,
    COUNT(DISTINCT o.id) AS pedidos_realizados,
    ROUND(SUM(p.importe), 2) AS total_pagado_exitoso
FROM channels ch
JOIN customers cust ON ch.id = cust.canal_id
LEFT JOIN orders o ON cust.id = o.cliente_id
LEFT JOIN payments p ON o.id = p.pedido_id
LEFT JOIN payment_status ps ON p.estado_pago_id = ps.id
WHERE ps.estado = 'Aceptado' OR p.id IS NULL
GROUP BY ch.canal
ORDER BY total_pagado_exitoso DESC;
"""
df_canales = pd.read_sql_query(query_5, conn)
df_canales


,canal_adquisicion,clientes_captados,pedidos_realizados,total_pagado_exitoso
0,paid ads,1,0,None
1,organic,3,0,None
2,empresa,7,0,None


### Resolver cuestiones
- ¿Por qué `unit_price` está en `order_items` y no se lee directamente de `products.price`?
    - porque el producto puede sufrir un cambio en su valor a lo largo del tiempo, que no deberia de verse reflejado en los pedidos ya realizados
- ¿Por qué `country` está directamente en `customers` y no es una tabla `countries` separada?
    - porque es un campo estatico o que varie poco, en cualquier momento puede haber un nuevo cliente de otro pais y entonce habria que actualizar 2 tablas, ademas de consumir espacio de manera innecesaria
- Si en `orders` almacenásemos el nombre del cliente (`customer_name`) además de `customer_id`, ¿qué forma normal se violaría y por qué?
    - La tercera forma normal ya que ningun atributo no-clave puede depender de otro no-clave. Si no dependiera entonces tendriamos grupos repetidos y eso violaria la primera forma normal